# 📊 Smart Business Dashboard — Exploratory Data Analysis
**Author:** Harsh Sachan | Oracle Certified Data Scientist

This notebook walks through the full EDA pipeline used to power the AI Insights engine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded ✅')

## 1. Load Data

In [ ]:
sales    = pd.read_csv('../data/sales_data.csv', parse_dates=['date'])
customers= pd.read_csv('../data/customer_data.csv')
churn    = pd.read_csv('../data/churn_data.csv')

print(f'Sales:     {sales.shape}')
print(f'Customers: {customers.shape}')
print(f'Churn:     {churn.shape}')

## 2. Sales EDA

In [ ]:
# Monthly revenue trend
sales['month'] = sales['date'].dt.to_period('M')
monthly = sales.groupby('month')['revenue'].sum().reset_index()
monthly['month'] = monthly['month'].dt.to_timestamp()

plt.figure(figsize=(14,4))
plt.plot(monthly['month'], monthly['revenue']/1e6, color='#58a6ff', linewidth=2)
plt.fill_between(monthly['month'], monthly['revenue']/1e6, alpha=0.15, color='#58a6ff')
plt.title('Monthly Revenue (₹M)', fontsize=14)
plt.xlabel('Month'); plt.ylabel('Revenue (₹M)')
plt.tight_layout(); plt.show()

In [ ]:
# Revenue by region
fig, axes = plt.subplots(1, 2, figsize=(14,4))
sales.groupby('region')['revenue'].sum().sort_values().plot(kind='barh', ax=axes[0], color='#3fb950')
axes[0].set_title('Revenue by Region')
sales.groupby('product')['revenue'].sum().sort_values().plot(kind='barh', ax=axes[1], color='#bc8cff')
axes[1].set_title('Revenue by Product')
plt.tight_layout(); plt.show()

## 3. Churn EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Churn by region
churn.groupby('region')['churned'].mean().sort_values().plot(kind='bar', ax=axes[0], color='#f85149')
axes[0].set_title('Churn Rate by Region'); axes[0].set_ylabel('Churn Rate')

# NPS vs Churn
churn.groupby('churned')['nps_score'].plot(kind='hist', alpha=0.7, ax=axes[1], bins=10)
axes[1].set_title('NPS Score Distribution by Churn'); axes[1].legend(['Retained','Churned'])

# Last login vs Churn
churn.boxplot(column='last_login_days', by='churned', ax=axes[2])
axes[2].set_title('Last Login Days vs Churn'); axes[2].set_xlabel('Churned (0=No, 1=Yes)')

plt.tight_layout(); plt.show()

## 4. Correlation Heatmap

In [ ]:
numeric_cols = ['age','annual_spend','support_tickets','nps_score','products_owned','last_login_days','churned']
corr = churn[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation with Churn')
plt.tight_layout(); plt.show()

## 5. Feature Importance (from ML Model)

In [ ]:
feat_imp = pd.read_csv('../models/feature_importance.csv')

plt.figure(figsize=(8,4))
bars = plt.barh(feat_imp['feature'], feat_imp['importance']*100, color='#bc8cff')
plt.xlabel('Importance (%)')
plt.title('Random Forest Feature Importance — Churn Prediction')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()
print(feat_imp.to_string(index=False))

## ✅ Summary

- **Sales drop** detected in North, South, West, Central regions
- **Churn rate** of 39.5% — highest in East (50.5%)
- **Top churn predictors**: Last Login Days, Annual Spend, Support Tickets
- **Product C** is the top performer; Product A needs attention
- **Avg profit margin**: 43.6% — healthy but East region has untapped potential